# FWER calibration — verification across detectors and scores

A detector fires when a penalised score exceeds zero. The default (BIC-style) penalties
depend on *(n_samples, n_features)* only, and on short series they can admit far more
false alarms than intended.

**Goal.** Choose `penalty_scale` so that the **family-wise error rate** (FWER) — the
probability of flagging at least one change on change-free data — equals a target
`level` (here 10 %).

## How calibration works

The procedure is the same for every detector:

1. **Simulate the null.** Draw `N_SIMS` change-free data sets from a null model — either
   parametric Gaussian draws or resampling from a clean pool.
2. **Find the critical scale of each data set.** For a given data set, let `c_b` be the
   smallest `penalty_scale` at which the detector reports no changepoints. Below `c_b`
   it raises at least one false alarm; at or above it it is silent.
3. **Take a quantile.** Set `penalty_scale` to the `(1 − level)` empirical quantile of
   `{c_b}`. By construction a fraction `level` of null data sets still fire, which is the
   FWER we are targeting.

Only step 2 depends on the detector. How cheaply we can find `c_b` is governed by the
detector's structure, which is what the three **calibration strategies** address. The
strategy is read from the detector's `_calibration_strategy` attribute; it is not set by
the user.

| Strategy | Detectors | Fits per null sample |
|---|---|---|
| `"max_score"` | MovingWindow, SBS, CBS | 1 |
| `"path_search"` | PELT | ~3–6 |
| `"detection_count"` | CAPA, and the general fallback | ~15–25 |

- **`max_score` (closed form).** A scan-and-threshold detector fires precisely when some
  window's score exceeds the penalty, so the critical scale is `max(score) / base`,
  obtained from a single fit. Exact and fastest.
- **`path_search` (PELT).** PELT optimises jointly over all changepoint sets, so the
  binding constraint is not the single best split but `β* = max_k G_k / k`, the largest
  average cost reduction per changepoint. A secant search on the convex hull of cost
  against `k` locates `β*` exactly in a few fits.
- **`detection_count` (general fallback).** Bracket and bisect `penalty_scale` by
  refitting the detector and checking whether it still fires. It assumes nothing about
  the detector beyond a single penalty knob, so it applies to any detector — including
  ones such as CAPA whose penalty has no single scalar `base`. Slowest, but exact, since
  it runs the detector itself.

Both calibration and the empirical-FWER evaluation run in parallel (`n_jobs`); each cell
reports its calibration and evaluation time.

**Sections.**
- **A** — two scan-and-threshold detectors (MovingWindow, SBS) × four change scores; `max_score`.
- **B** — one mean-shift statistic across detector families; `max_score` and CAPA's `detection_count`.
- **C** — `detection_count` forced on selected combinations, as a check of the general fallback.
- **D** — PELT × costs with `path_search`, plus the level–FWER calibration curve.

## 0. Setup

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from sklearn.base import clone

from skchange.new_api.detectors import (
    CAPA,
    PELT,
    CircularBinarySegmentation,
    MovingWindow,
    SeededBinarySegmentation,
)
from skchange.new_api.interval_scorers import (
    CUSUM,
    ContinuousLinearTrendScore,
    GaussianCost,
    L2Cost,
    L2Saving,
    L2TransientScore,
    MultivariateGaussianScore,
    RankScore,
)
from skchange.new_api.tuning import calibrate_penalty_scale

# ---- Central parameters (reduce N_SIMS / N_EVAL for a quick smoke test) ----
N = 50  # series length
P = 2  # number of features
LEVEL = 0.1  # target FWER

N_SIMS = 10_000  # null simulations for calibration
N_EVAL = 10_000  # independent null series for empirical-FWER check
N_CALIB_POOL = 10 * N  # resampling pool: 10x N rows of clean data

N_JOBS = 6  # all cores -- used for BOTH calibration AND evaluation

RNG = np.random.default_rng(0)

# Reference null series: anchors the null-sample shape and the base penalty
X_ref = RNG.normal(size=(N, P))

# Larger clean pool for the permutation (resampling) null model.
# Drawing N rows from a 10N pool gives the sampler access to fresh extremes.
X_calib_pool = RNG.normal(size=(N_CALIB_POOL, P))


def _fires_on_null(detector, seed, n, p):
    """Fit a fresh clone on one N(0,1) null series; 1 if it fires, else 0."""
    rng = np.random.default_rng(seed)
    X_null = rng.normal(size=(n, p))
    fitted = clone(detector).fit(X_null)
    return int(len(fitted.predict_changepoints(X_null)) > 0)


def empirical_fwer(detector, n=N, p=P, seed=99, n_eval=N_EVAL, n_jobs=N_JOBS):
    """Fraction of change-free series on which the detector fires at least once.

    Parallelised across the ``n_eval`` independent series. Each series draws
    from its own ``SeedSequence`` child, so the result is reproducible and
    invariant to ``n_jobs`` (the same construction the calibrator uses).
    """
    child_seeds = np.random.SeedSequence(seed).spawn(n_eval)
    alarms = Parallel(n_jobs=n_jobs)(
        delayed(_fires_on_null)(detector, s, n, p) for s in child_seeds
    )
    return sum(alarms) / n_eval


def calibrate_and_eval(detector, sampler="gaussian", seed=0, calibration_strategy=None):
    """Calibrate ``penalty_scale`` with the given null model, then measure FWER.

    Pass ``calibration_strategy`` to override the detector's own strategy (for
    example, to force ``"detection_count"`` on a detector that defaults to
    ``"max_score"``). Returns a dict with the calibrated ``scale``, the
    empirical ``fwer``, and the wall-clock ``t_calib`` / ``t_eval`` in seconds.
    """
    calib_kw = dict(
        sampler=sampler,
        level=LEVEL,
        n_simulations=N_SIMS,
        random_state=seed,
        n_jobs=N_JOBS,
    )
    if sampler == "permutation":
        calib_kw["X_calib"] = X_calib_pool
    if calibration_strategy is not None:
        calib_kw["calibration_strategy"] = calibration_strategy

    t0 = time.perf_counter()
    scale = calibrate_penalty_scale(detector, X_ref, **calib_kw)
    t_calib = time.perf_counter() - t0

    fitted = clone(detector).set_params(penalty_scale=scale)
    t1 = time.perf_counter()
    fwer = empirical_fwer(fitted)
    t_eval = time.perf_counter() - t1

    return {"scale": scale, "fwer": fwer, "t_calib": t_calib, "t_eval": t_eval}


print("Setup complete.")
print(f"  N={N}, P={P}, LEVEL={LEVEL:.0%}")
print(f"  N_SIMS={N_SIMS:,}, N_EVAL={N_EVAL:,}, N_CALIB_POOL={N_CALIB_POOL:,}")
print(f"  n_jobs={N_JOBS}  (calibration AND evaluation)")

## Section A — Fast: detectors × change scores

Vary the change score across two scan-and-threshold detectors, `MovingWindow` and
`SeededBinarySegmentation` (SBS). Both use the `"max_score"` strategy (one fit per null
sample).

Null models:
- **Gaussian**: draws i.i.d. N(0,1) null samples. Parametric; fastest.
- **Permutation**: resamples `N` rows from a clean `10×N`-row pool. Non-parametric;
  more honest when the null distribution is unknown.

**Expected outcome:** empirical FWER ≈ 10 % for every detector × score combination and
both null models — with **one instructive exception**, `MovingWindow + MultivariateGaussian`,
explained in the note below the table.

In [ ]:
SEC_A_SCORES = {
    "CUSUM": CUSUM(),
    "LinearTrend": ContinuousLinearTrendScore(),
    "MultivariateGaussian": MultivariateGaussianScore(),
    "Rank": RankScore(),
}
SEC_A_DETECTORS = {
    "MovingWindow": MovingWindow,
    "SBS": SeededBinarySegmentation,
}

results_a = {}
for det_name, det_cls in SEC_A_DETECTORS.items():
    for score_name, score in SEC_A_SCORES.items():
        det = det_cls(change_score=score)
        results_a[(det_name, score_name)] = {}
        for sampler in ("gaussian", "permutation"):
            r = calibrate_and_eval(det, sampler)
            results_a[(det_name, score_name)][sampler] = r
            print(
                f"  {det_name:<13s}({score_name:<20s}) {sampler:<11s}"
                f"  scale={r['scale']:.3f}  FWER={r['fwer']:.1%}  (target {LEVEL:.0%})"
                f"  [calib {r['t_calib']:5.1f}s | eval {r['t_eval']:5.1f}s]"
            )

In [ ]:
# Summary table for Section A
print(f"\nSection A - detector x score  (target FWER = {LEVEL:.0%})")
header = (
    f"{'Detector':<13s} {'Score':<22s} {'Gauss FWER':>11s} {'Perm FWER':>11s} "
    f"{'calib s':>9s} {'eval s':>8s}"
)
print(header)
print("-" * len(header))
for (det_name, score_name), res in results_a.items():
    g, p = res["gaussian"], res["permutation"]
    ok_g = "ok" if abs(g["fwer"] - LEVEL) < 0.015 else "<<"
    ok_p = "ok" if abs(p["fwer"] - LEVEL) < 0.015 else "<<"
    t_cal = g["t_calib"] + p["t_calib"]
    t_evl = g["t_eval"] + p["t_eval"]
    print(
        f"{det_name:<13s} {score_name:<22s} {g['fwer']:>8.1%} {ok_g:<2s} "
        f"{p['fwer']:>8.1%} {ok_p:<2s} {t_cal:>9.1f} {t_evl:>8.1f}"
    )

### `MovingWindow + MultivariateGaussian`: conservativeness at small `N`

Under `max_score` this combination attains an empirical FWER well below target at
`N = 50` (roughly half), while every other combination in the table is on target. The
gap narrows as `N` grows and is negligible at the sample sizes one usually calibrates
for — it is a small-sample effect.

It is specific to `MovingWindow`'s selection step rather than to the score or the
calibration arithmetic: `SBS` with the same score is on target, and `max_score` returns
the exact `max(score) / base`. A plausible mechanism is that `MovingWindow` retains a
candidate split only if its score is a local maximum within a bandwidth-sized window,
and near the series boundary that window extends past the last valid split position
(padded with `NaN`), so a genuine boundary peak can be discarded. The
`MultivariateGaussian` score requires `n_features + 1` points on each side of a split,
which shrinks the valid range and makes boundary windows more common at small `N`. This
is consistent with the evidence but we have not isolated it conclusively.

Either way, calibrating against the detector itself rather than the closed-form score
maximum — the `detection_count` strategy (Section C) — attains the target level.

## Section B — One mean-shift statistic across detector families

Each detector consumes the same underlying L2 mean-shift statistic through a different
score family, which checks that calibration holds across detector families and not only
across scores.

| Detector | Score family | L2 representative | Strategy |
|---|---|---|---|
| MovingWindow | `change_score` | `CUSUM` | `max_score` |
| SeededBinarySegmentation | `change_score` | `CUSUM` | `max_score` |
| CircularBinarySegmentation | `transient_score` | `L2TransientScore` | `max_score` |
| CAPA | `saving` (segment) | `L2Saving` | `detection_count` |

The first three use the closed-form `max_score` (one fit). CAPA's penalty has no single
scalar `base` — it carries separate segment and point penalties — so it uses
`detection_count` instead, which is why its calibration is markedly slower below.

**Expected outcome:** empirical FWER ≈ 10 % for every detector and both null models.

In [ ]:
SEC_B_DETECTORS = {
    "MovingWindow(CUSUM)": MovingWindow(change_score=CUSUM()),
    "SBS(CUSUM)": SeededBinarySegmentation(change_score=CUSUM()),
    "CBS(L2TransientScore)": CircularBinarySegmentation(
        transient_score=L2TransientScore()
    ),
    "CAPA(L2Saving)": CAPA(segment_saving=L2Saving()),
}

results_b = {}
for det_name, det in SEC_B_DETECTORS.items():
    results_b[det_name] = {}
    for sampler in ("gaussian", "permutation"):
        r = calibrate_and_eval(det, sampler)
        results_b[det_name][sampler] = r
        print(
            f"  {det_name:<24s} {sampler:<11s}"
            f"  scale={r['scale']:.3f}  FWER={r['fwer']:.1%}  (target {LEVEL:.0%})"
            f"  [calib {r['t_calib']:5.1f}s | eval {r['t_eval']:5.1f}s]"
        )

In [ ]:
# Summary table for Section B
print(f"\nSection B - detector x L2 score  (target FWER = {LEVEL:.0%})")
header = (
    f"{'Detector':<24s} {'Gauss FWER':>11s} {'Perm FWER':>11s} "
    f"{'calib s':>9s} {'eval s':>8s}"
)
print(header)
print("-" * len(header))
for det_name, res in results_b.items():
    g, p = res["gaussian"], res["permutation"]
    ok_g = "ok" if abs(g["fwer"] - LEVEL) < 0.015 else "<<"
    ok_p = "ok" if abs(p["fwer"] - LEVEL) < 0.015 else "<<"
    t_cal = g["t_calib"] + p["t_calib"]
    t_evl = g["t_eval"] + p["t_eval"]
    print(
        f"{det_name:<24s} {g['fwer']:>8.1%} {ok_g:<2s} {p['fwer']:>8.1%} {ok_p:<2s} "
        f"{t_cal:>9.1f} {t_evl:>8.1f}"
    )

## Section C — The `detection_count` fallback on selected combinations

`detection_count` brackets and bisects `penalty_scale` by refitting the actual detector,
so it makes no use of the detector's internal structure and applies universally. Here we
force it on three combinations that would otherwise use `max_score`, to confirm it
attains the target level. One of them is `MovingWindow + MultivariateGaussian`, which
`max_score` leaves conservative at this sample size (Section A); calibrating against the
detector directly removes that gap.

**Expected outcome:** empirical FWER ≈ 10 % for every combination, including the one that
was conservative under `max_score`.

In [ ]:
SEC_C_COMBOS = {
    "SBS(CUSUM)": SeededBinarySegmentation(change_score=CUSUM()),
    "MovingWindow(MVGauss)": MovingWindow(change_score=MultivariateGaussianScore()),
    "CBS(L2TransientScore)": CircularBinarySegmentation(
        transient_score=L2TransientScore()
    ),
}

# Gaussian null only: detection_count refits the detector many times per sample,
# so it is slower than the max_score sections above.
results_c = {}
for name, det in SEC_C_COMBOS.items():
    r = calibrate_and_eval(
        det, sampler="gaussian", calibration_strategy="detection_count"
    )
    results_c[name] = r
    print(
        f"  {name:<22s} scale={r['scale']:.3f}  FWER={r['fwer']:.1%}"
        f"  (target {LEVEL:.0%})"
        f"  [calib {r['t_calib']:5.1f}s | eval {r['t_eval']:5.1f}s]"
    )

## Section D — PELT with the `path_search` strategy

PELT uses the exact convex-hull secant search (`path_search`) to find
`β* = max_k G_k / k`, the largest average cost reduction per changepoint, in ~3–6 fits
per null sample rather than the ~15–25 of bisection.

This section calibrates PELT with two cost functions and then plots the **calibration
curve**: empirical FWER against nominal level. Points on the diagonal indicate correct
calibration.

> **Note — slowest cell in the notebook.** The calibration curve re-calibrates from
> scratch at every level in `SWEEP_LEVELS` (5 levels × 2 costs = 10 full calibrations of
> `N_SIMS` simulations each) and then evaluates the FWER `N_EVAL` times per point. Expect
> a few minutes at the default settings. To shorten it, reduce `SWEEP_LEVELS` or
> `N_SIMS` / `N_EVAL`.

In [ ]:
SEC_D_COSTS = {
    "PELT(L2Cost)": PELT(cost=L2Cost()),
    "PELT(GaussianCost)": PELT(cost=GaussianCost()),
}

# Point-estimate calibration at LEVEL (Gaussian null only for PELT -- slowest section)
results_d = {}
for det_name, det in SEC_D_COSTS.items():
    r = calibrate_and_eval(det, sampler="gaussian")
    results_d[det_name] = r
    print(
        f"  {det_name:<22s}  scale={r['scale']:.3f}  FWER={r['fwer']:.1%}"
        f"  (target {LEVEL:.0%})"
        f"  [calib {r['t_calib']:5.1f}s | eval {r['t_eval']:5.1f}s]"
    )

print(f"\nPELT calibration strategy: {PELT._calibration_strategy}")

In [ ]:
# ---- Calibration curve: empirical FWER vs nominal level ----
SWEEP_LEVELS = [0.01, 0.05, 0.10, 0.20, 0.30]

curve_results = {}
for det_name, det in SEC_D_COSTS.items():
    empirical_fwers = []
    for lv in SWEEP_LEVELS:
        t0 = time.perf_counter()
        scale = calibrate_penalty_scale(
            det,
            X_ref,
            sampler="gaussian",
            level=lv,
            n_simulations=N_SIMS,
            random_state=42,
            n_jobs=N_JOBS,
        )
        t_calib = time.perf_counter() - t0
        fitted = clone(det).set_params(penalty_scale=scale)
        t1 = time.perf_counter()
        fwer = empirical_fwer(fitted)
        t_eval = time.perf_counter() - t1
        empirical_fwers.append(fwer)
        print(
            f"  {det_name:<22s} level={lv:.0%}  FWER={fwer:.1%}"
            f"  [calib {t_calib:5.1f}s | eval {t_eval:5.1f}s]"
        )
    curve_results[det_name] = empirical_fwers

# Plot
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 0.35], [0, 0.35], "k--", lw=1, label="ideal (y = x)")
for det_name, fwers in curve_results.items():
    ax.plot(SWEEP_LEVELS, fwers, "o-", label=det_name)
ax.set_xlabel("Nominal level")
ax.set_ylabel("Empirical FWER")
ax.set_title("PELT calibration curve")
ax.legend()
plt.tight_layout()
plt.show()